In [ ]:
import polars as pl
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
from analyses.rust import contact
from pathlib import Path

In [ ]:
latpaths = {}
for replica in Path("../runs/cil_med-act/energy-10/").iterdir():
    rep = int(replica.name)
    latpaths[rep] = []
    for file in replica.joinpath("lattices").iterdir():
        latpaths[rep].append((
            str(file),
            str(file).replace("lattices", "act_lattices")
        ))

In [ ]:
def acts_to_df(acts):
    return pl.DataFrame({
        "label": ["c"] * len(acts.cell) + ["m"] * len(acts.medium) + ["s"] * len(acts.solid),
        "act": acts.cell + acts.medium + acts.solid,
    })

In [ ]:
klats = []
dfs = []
for r in range(0, 17):
    clat, alat = latpaths[0][100]
    res = contact.kernel_act(clat, alat, 500, 500, r, True)
    klat = res[1]
    klat = np.array(klat).reshape(500, 500)
    klats.append(klat)
    dfs.append(acts_to_df(res[0]).with_columns(radius=r))
klat_3d = np.array(klats)
rdf = pl.concat(dfs)

In [ ]:
px.imshow(klat_3d, animation_frame=0, width=800, height=800)

In [ ]:
px.violin(rdf, y="act", color="label", animation_frame="radius")

In [ ]:
cdf = rdf.group_by(["radius", "label"]).agg(
    mean=pl.col("act").mean()
).sort(
    "radius"
).pivot(
    on="label",
    index="radius", 
    values="mean"
).with_columns(
    diff=pl.col("m") - pl.col("c"),
    mdiff=pl.col("m") / pl.col("c")
).unpivot(
    index="radius"
)
px.bar(cdf, x="radius", y="value", color="variable", barmode="group")

In [ ]:


lcdf = acts_to_df(contact.kernel_act(*(latpaths[0][100]), 500, 500, 1))
# lcdf = lcdf.filter(pl.col("act") != 0)
print(lcdf.group_by("label").agg([pl.col("act").mean().alias("mean_act"), pl.col("act").count().alias("count")]))
px.violin(lcdf, y="act", color="label", box=True)

In [ ]:
_neighs = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]


def inbounds(x, y, w, h):
    if x < 0:
        return False
    if x > w - 1:
        return False
    if y < 0:
        return False
    if y > h - 1:
        return False
    return True


def neighs(x, y, w, h):
    return filter(lambda pos: inbounds(pos[0], pos[1], w, h), map(lambda pos: (pos[0] + x, pos[1] + y), _neighs))


# This is not quite what ava did.
# Try:
#   Using the geom mean of each pos
#   Taking the average per neighbour of the entire cell (use two NCELLS X NCELLS matrices to calculate the averages)
def act_sum_local(lat, act):
    latdf = pl.read_parquet(lat)
    actdf = pl.read_parquet(act)
    labels = []
    actvs = []
    for i in range(0, latdf.shape[0]):
        for j in range(0, latdf.shape[1]):
            for n in neighs(i, j, latdf.width, latdf.height):
                ij_sigma = latdf[i, j]
                if not ij_sigma.isnumeric():
                    continue
                n_sigma = latdf[*n]
                if ij_sigma == n_sigma:
                    continue
                act = actdf[i, j]
                if n_sigma.isnumeric():
                    lab = "c"
                else:
                    lab = n_sigma
                actvs.append(act)
                labels.append(lab)
                
    return pl.DataFrame({
        "label": labels,
        "act": actvs
    })


def act_sum_interface(latdf: pl.DataFrame, actdf: pl.DataFrame):
    cell1 = []
    cell2 = []
    actvs = []
    for i in range(0, latdf.shape[0]):
        for j in range(0, latdf.shape[1]):
            for n in neighs(i, j, latdf.width, latdf.height):
                ij_sigma = latdf[i, j]
                if not ij_sigma.isnumeric():
                    continue
                n_sigma = latdf[*n]
                if ij_sigma == n_sigma:
                    continue
                act = 200 * abs(geom_mean((i, j), latdf, actdf) - geom_mean(n, latdf, actdf))
                cell1.append(ij_sigma)
                cell2.append(n_sigma)
                actvs.append(act)
    df = pl.DataFrame({
        "cell1": cell1,
        "cell2": cell2,
        "act": actvs
    })
    df = df.group_by(["cell1", "cell2"]).agg(pl.col("act").mean())
    
    return df.with_columns(label=pl.Series("c" if x.isnumeric() and y.isnumeric() else y for x, y in zip(df["cell1"], df["cell2"]))).select(["label", "act"])


def act_sum_geom(latdf: pl.DataFrame, actdf: pl.DataFrame):
    labels = []
    actvs = []
    for i in range(0, latdf.shape[0]):
        for j in range(0, latdf.shape[1]):
            for n in neighs(i, j, latdf.width, latdf.height):
                ij_sigma = latdf[i, j]
                if not ij_sigma.isnumeric():
                    continue
                n_sigma = latdf[*n]
                if ij_sigma == n_sigma:
                    continue
                act = abs(geom_mean((i, j), latdf, actdf) - geom_mean(n, latdf, actdf))
                if n_sigma.isnumeric():
                    lab = "c"
                else:
                    lab = n_sigma
                actvs.append(act)
                labels.append(lab)
                
    return pl.DataFrame({
        "label": labels,
        "act": actvs
    })
    

def geom_mean(pos, latdf, actdf):
    pos_spin = latdf[*pos]
    act_prod = actdf[*pos]
    act_count = 1
    for neigh in neighs(pos[0], pos[1], latdf.width, latdf.height):
        if latdf[*neigh] != pos_spin:
            continue
        act_prod *= actdf[*neigh]
        act_count += 1
    return act_prod ** (1 / act_count)

In [ ]:
px.imshow(pl.read_parquet(latpaths[0][100][1]), width=500, height=500)

In [ ]:
asdf = act_sum_local(*(latpaths[0][100]))

In [ ]:
print(asdf.group_by("label").agg([pl.col("act").mean().alias("mean_act"), pl.col("act").count().alias("count")]))
px.violin(asdf, y="act", color="label", box=True)

In [ ]:
asgdf = act_sum_geom(ldfs[4950000], adfs[4950000])

In [ ]:
print(asgdf.group_by("label").agg([pl.col("act").mean().alias("mean_act"), pl.col("act").count().alias("count")]))
px.violin(asgdf, y="act", color="label", box=True)

In [ ]:
# Geom mean does show larger values for cell-medium but has a large proportion of 0s
asgdf.filter(pl.col("act") == 0, pl.col("label") == "c").height / asgdf.filter(pl.col("label") == "c").height

In [ ]:
asidf = act_sum_interface(ldfs[4950000], adfs[4950000])

In [ ]:
print(asidf.group_by("label").agg([pl.col("act").mean().alias("mean_act"), pl.col("act").count().alias("count")]))
px.violin(asidf, y="act", color="label", box=True)